# Triphorium · 个人简易启动

**日常只需：打开此页 → 点击下面唯一代码格的 ▶ → 回平台生成。**

- 使用拥有 Colab 额度的账号。本文件直接从 GitHub 加载，不需要下载、上传或访问私人 Drive notebook。平台登录账号可以不同。
- 首次选择 A100 GPU。保留当前 ≥40GB 的实验性卸载配置；40GB 的生成速度、质量与稳定性不作保证，80GB 通常有更多余量。
- 首次在左侧 **钥匙 / Secrets** 添加 `CREW_SITE_TOKEN` 和 `CREW_H3_FAST_KEY`，打开本 notebook 的访问开关。值在平台设置的配对区获取；不要写进代码或聊天。之后启动自动读取，无需反复复制。
- 启动时不访问 Drive。需要归档时，再点击运行结果下方的 **同步到 Drive（可选）** 按钮；只有此时才请求归档账号授权。
- 本单元自动复用依赖、缓存文件和仍在运行的 Worker；新运行时仍需安装及下载。不会运行额外测试镜头、训练或覆盖 LoRA。
- 打开链接不会自动执行。Google 登录、首次授权、GPU 选择以及运行时回收后的重新启动仍需操作。
- **下班：Runtime → Disconnect and delete runtime。**关闭网页不等于停止 GPU 计费。


In [ ]:
#@title ▶ 启动 / 恢复 Triphorium（只运行这一格）
import io,json,time,sys,os,subprocess,threading,getpass,shutil
from pathlib import Path
import requests
from google.colab import userdata
SITE='https://triphorium-team-studio.triphorium.chatgpt.site'
print('1/4 · 检查 GPU 与凭据',flush=True)
if not shutil.which('nvidia-smi'):
 raise RuntimeError('当前是 CPU 运行时。请在 Runtime → Change runtime type 选择 A100，再运行本单元。')

def secret(name):
 try:
  value=userdata.get(name)
 except Exception:
  print('首次配对：请在左侧钥匙 Secrets 添加 '+name+' 并允许本 notebook 访问；也可本次临时输入。',flush=True)
  value=getpass.getpass(name+': ')
 if not value or not value.strip():raise RuntimeError('未提供 '+name+'；未启动生成。')
 return value.strip()
site_token=secret('CREW_SITE_TOKEN')
bridge_key=secret('CREW_H3_FAST_KEY')
print('2/4 · 恢复引擎。新运行时需要安装和下载；同一运行时会复用。',flush=True)
gpu=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],text=True).strip()
vram_mib=int(gpu.splitlines()[0].split(',')[1])
assert vram_mib>=39000,'当前 FastH3 配置需要至少 40GB 显存；请选择 A100。此时尚未下载模型。'
print('40GB 分阶段卸载验证模式' if vram_mib<80000 else '80GB 模式')
print(gpu)
C=Path('/content/ComfyUI');PY='/content/crew-fast-env/bin/python'
def run(args):
 p=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
 print(p.stdout[-12000:],flush=True)
 p.check_returncode()
if not C.exists():run(['git','clone','https://github.com/Comfy-Org/ComfyUI.git',str(C)])
run(['git','-C',str(C),'checkout','c5751c81abe131202eea9e4c66d000589013dd0b'])
if not Path('/content/crew-fast-env/bin/pip').exists():
 run([sys.executable,'-m','pip','install','virtualenv'])
 run([sys.executable,'-m','virtualenv','--system-site-packages','/content/crew-fast-env'])
marker=Path('/content/crew-fast-env/.triphorium-ready-v1')
if not marker.exists():
 run([PY,'-m','pip','install','-r',str(C/'requirements.txt'),'huggingface_hub','requests','Pillow'])
 marker.touch()
else:print('复用本运行时已安装的依赖',flush=True)
download_code="""from huggingface_hub import hf_hub_download
jobs=[('FastVideo/FastVideo-FastH3-Comfy','ec1e3aa374a91c57b0b94a1623b7e657c0498cf2','diffusion_models/fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors'),('Comfy-Org/MiniMax-H3','54472221d27c695fca9621b0c3ed52f06c4af707','text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors'),('Comfy-Org/MiniMax-H3','54472221d27c695fca9621b0c3ed52f06c4af707','vae/minimax_h3_video_vae_fp16.safetensors')]
for repo,rev,name in jobs:hf_hub_download(repo,name,revision=rev,local_dir='/content/ComfyUI/models')
"""
run([PY,'-c',download_code])
try:
 requests.get('http://127.0.0.1:8188/system_stats',timeout=3).raise_for_status()
 print('复用已运行 ComfyUI；下一步会验证模型与节点')
except requests.RequestException:
 log=open('/content/crew-fast-server.log','a')
 comfy_proc=subprocess.Popen([PY,str(C/'main.py'),'--listen','127.0.0.1','--port','8188','--disable-auto-launch','--lowvram','--reserve-vram','4'],cwd=C,stdout=log,stderr=subprocess.STDOUT)
 for _ in range(120):
  try:requests.get('http://127.0.0.1:8188/system_stats',timeout=3).raise_for_status();break
  except requests.RequestException:
   if comfy_proc.poll() is not None:raise RuntimeError('ComfyUI 启动失败，请查看 /content/crew-fast-server.log')
   time.sleep(3)
 else:raise TimeoutError('ComfyUI 未就绪')
print('COMFY_READY')


print('3/4 · 验证平台连接并启动 Worker',flush=True)
worker_source='"""FastH3 production bridge. Credentials supplied on stdin, never stored in code.\nRequires the verified FastH3 V2 ComfyUI instance on localhost:8188.\n"""\nimport hashlib,json,os,pathlib,shutil,sys,threading,time,uuid\nimport requests\nfrom PIL import Image\nPROFILE=\'h3-fast\'\nMODEL=\'FastH3 V2 · 8-step\'\nROOT=pathlib.Path(\'/content/ComfyUI\')\nSITE=\'https://triphorium-team-studio.triphorium.chatgpt.site\'\nCOMFY=\'http://127.0.0.1:8188\'\nENGINE=\'FastH3 V2 INT8 · 8 steps · VSA 10% · no LoRA · silent\'\nMETA={\'node\':PROFILE,\'model\':MODEL,\'engine\':ENGINE,\'capabilities\':[\'clips\',PROFILE]}\n\ndef graph(image,prompt,seed,width,height,prefix):\n    def n(kind,**kw):return {\'class_type\':kind,\'inputs\':kw}\n    return {\n      \'1\':n(\'UNETLoader\',unet_name=\'fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors\',weight_dtype=\'default\'),\n      \'2\':n(\'CLIPLoader\',clip_name=\'qwen3vl_32b_minimax_h3_int8_convrot.safetensors\',type=\'minimax\',device=\'default\'),\n      \'3\':n(\'VAELoader\',vae_name=\'minimax_h3_video_vae_fp16.safetensors\'),\n      \'5\':n(\'LoadImage\',image=image),\n      \'6\':n(\'MiniMaxH3ImageToVideo\',clip=[\'2\',0],vae=[\'3\',0],prompt=prompt,width=width,height=height,length=124,first_frame=[\'5\',0]),\n      \'17\':{\'class_type\':\'BlockSparseAttention\',\'inputs\':{\'model\':[\'1\',0],\'selection\':\'vsa\',\'selection.keep_percent\':10,\'start_percent\':0.2,\'end_percent\':1,\'dense_blocks\':\'\',\'min_tokens\':12288,\'extra_tokens\':256,\'sink_conditioning\':\'exact_kv_and_rows\',\'verbose\':False}},\n      \'18\':n(\'MiniMaxH3SigmaShift\',model=[\'17\',0],shift_video=10,shift_audio=3),\n      \'7\':n(\'BasicGuider\',model=[\'18\',0],conditioning=[\'6\',0]),\n      \'8\':n(\'RandomNoise\',noise_seed=seed),\'9\':n(\'KSamplerSelect\',sampler_name=\'res_multistep\'),\n      \'10\':n(\'BasicScheduler\',model=[\'18\',0],scheduler=\'simple\',steps=8,denoise=1),\n      \'11\':n(\'SamplerCustomAdvanced\',noise=[\'8\',0],guider=[\'7\',0],sampler=[\'9\',0],sigmas=[\'10\',0],latent_image=[\'6\',1]),\n      \'12\':n(\'VAEDecode\',samples=[\'11\',0],vae=[\'3\',0]),\n      \'14\':n(\'CreateVideo\',images=[\'12\',0],fps=24),\n      \'15\':{\'class_type\':\'SaveVideo\',\'inputs\':{\'video\':[\'14\',0],\'filename_prefix\':prefix,\'format\':\'mp4\',\'format.codec\':\'h264\'}}}\n\nclass Worker:\n    def __init__(self,credentials):\n        self.cloud=requests.Session();self.cloud.headers.update({\'OAI-Sites-Authorization\':\'Bearer \'+credentials[\'site_token\'],\'x-crew-bridge-key\':credentials[\'bridge_key\']})\n        self.job=None;self.stop=threading.Event();self.alive=True\n    def post(self,route,payload):\n        r=self.cloud.post(SITE+\'/api/bridge/\'+route,json=payload,timeout=30);r.raise_for_status();return r.json()\n    def heartbeat(self):\n        while not self.stop.wait(20):\n            try:self.post(\'heartbeat\',{**META,**({\'id\':self.job[\'id\'],\'lease\':self.job[\'lease\']} if self.job else {})})\n            except Exception:pass\n    def progress(self,label,stage,shot,total,elapsed=0):\n        self.post(\'progress\',{\'id\':self.job[\'id\'],\'lease\':self.job[\'lease\'],\'type\':\'start\',\'from\':\'P\',\'to\':\'P\',\'label\':label,\'quiet\':True,\'metrics\':{\'stage\':stage,\'shot\':shot,\'shots\':total,\'elapsed\':elapsed}})\n    def run_job(self,job):\n        assert job[\'videoProfile\']==PROFILE and job[\'stage\']==\'clips\'\n        started=time.monotonic();assets=list(job.get(\'partialAssets\',[]));done={a[\'sourceId\'] for a in assets}\n        for i,a in enumerate(job[\'approvedImages\'],1):\n            if a[\'id\'] in done:continue\n            tick=time.monotonic();self.progress(\'FastH3 · 下载已审核设定图\',\'download\',i,len(job[\'approvedImages\']))\n            assert a[\'url\'].startswith(\'/api/media/\'), \'Invalid image URL\'\n            r=self.cloud.get(SITE+a[\'url\'].replace(\'/api/media/\',\'/api/bridge/media/\',1),timeout=60);r.raise_for_status()\n            name=\'crew-\'+str(uuid.uuid4())+\'.png\';dest=ROOT/\'input\'/name;dest.parent.mkdir(exist_ok=True);dest.write_bytes(r.content)\n            with Image.open(dest) as im:\n                w,h=im.size;scale=min(1280/max(w,h),(1280*704/(w*h))**0.5);w=max(64,round(w*scale/32)*32);h=max(64,round(h*scale/32)*32)\n                im.convert(\'RGB\').save(dest)\n            prompt=str(a.get(\'prompt\') or job[\'text\'])[:5000]\n            seed=int(hashlib.sha256((job[\'id\']+a[\'id\']+str(job.get(\'regen\',{}).get(a[\'id\'],0))).encode()).hexdigest()[:8],16)\n            prefix=\'crew/\'+job[\'id\']+\'/\'+str(uuid.uuid4())\n            rr=requests.post(COMFY+\'/prompt\',json={\'prompt\':graph(name,prompt,seed,w,h,prefix)},timeout=30);rr.raise_for_status();pid=rr.json()[\'prompt_id\']\n            self.progress(\'FastH3 · ComfyUI 已接收镜头\',\'sample\',i,len(job[\'approvedImages\']))\n            while True:\n                time.sleep(3);rr=requests.get(COMFY+\'/history/\'+pid,timeout=30);rr.raise_for_status();entry=rr.json().get(pid)\n                if entry:\n                    if entry.get(\'status\',{}).get(\'status_str\')==\'error\':raise RuntimeError(\'ComfyUI generation failed; inspect local execution log\')\n                    outputs=entry.get(\'outputs\',{}).get(\'15\',{});files=outputs.get(\'images\',outputs.get(\'videos\',[]))\n                    if files:break\n                    if entry.get(\'status\',{}).get(\'completed\'):raise RuntimeError(\'No video output\')\n                if time.monotonic()-tick>1800:\n                    requests.post(COMFY+\'/interrupt\',timeout=10)\n                    raise TimeoutError(\'Generation exceeded 30 minutes\')\n            f=files[0];video=(ROOT/\'output\'/f.get(\'subfolder\',\'\')/f[\'filename\']).resolve()\n            assert video.is_relative_to((ROOT/\'output\').resolve())\n            import subprocess\n            seconds=float(subprocess.check_output([\'ffprobe\',\'-v\',\'error\',\'-show_entries\',\'format=duration\',\'-of\',\'default=nw=1:nk=1\',str(video)],text=True))\n            self.progress(\'视频已生成 · 正在保存到团队界面\',\'upload\',i,len(job[\'approvedImages\']))\n            aid=str(uuid.uuid4())+\'.mp4\';raw=video.read_bytes()\n            rr=self.cloud.post(SITE+\'/api/bridge/asset\',params={\'id\':job[\'id\'],\'lease\':job[\'lease\'],\'asset\':aid},data=raw,headers={\'Content-Type\':\'video/mp4\'},timeout=120);rr.raise_for_status()\n            asset={**rr.json(),\'sourceId\':a[\'id\'],\'title\':a.get(\'title\',\'镜头 \'+str(i)),\'prompt\':prompt,\'seconds\':seconds,\'seed\':seed,\'engine\':ENGINE,\'profile\':PROFILE,\'timings\':{\'total\':round(time.monotonic()-tick,2)}}\n            self.post(\'checkpoint\',{\'id\':job[\'id\'],\'lease\':job[\'lease\'],\'asset\':asset});assets.append(asset)\n        self.post(\'result\',{\'id\':job[\'id\'],\'lease\':job[\'lease\'],\'owner\':\'P\',\'result\':{\'by\':\'model\',\'model\':MODEL,\'engine\':ENGINE,\'assets\':assets,\'seconds\':round(time.monotonic()-started,2),\'decision\':\'FastH3 视频已返回，请逐个播放审核。未对外发布。\'}})\n    def run(self):\n        rr=requests.get(COMFY+\'/object_info\',timeout=30);rr.raise_for_status();info=rr.json()\n        for node in [\'MiniMaxH3ImageToVideo\',\'MiniMaxH3SigmaShift\',\'BlockSparseAttention\']:\n            assert node in info, \'Missing ComfyUI node: \'+node\n        for loader,key,name in [(\'UNETLoader\',\'unet_name\',\'fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors\'),(\'CLIPLoader\',\'clip_name\',\'qwen3vl_32b_minimax_h3_int8_convrot.safetensors\'),(\'VAELoader\',\'vae_name\',\'minimax_h3_video_vae_fp16.safetensors\')]:\n            assert name in info[loader][\'input\'][\'required\'][key][0], \'Missing model: \'+name\n        self.post(\'heartbeat\',META);threading.Thread(target=self.heartbeat,daemon=True).start();print(\'FASTH3_BRIDGE_ONLINE\',flush=True)\n        while not self.stop.is_set():\n            try:\n                self.job=self.post(\'poll\',META).get(\'job\')\n                if self.job:\n                    try:self.run_job(self.job);print(\'COMPLETED\',self.job[\'id\'],flush=True)\n                    except Exception as e:\n                        self.post(\'result\',{\'id\':self.job[\'id\'],\'lease\':self.job[\'lease\'],\'error\':\'FastH3: \'+type(e).__name__+\'; inspect Colab worker log\'})\n                        print(\'JOB_ERROR\',type(e).__name__,flush=True)\n                    finally:self.job=None\n            except Exception as e:print(\'BRIDGE_RETRY\',type(e).__name__,flush=True)\n            self.stop.wait(8)\n\nif __name__==\'__main__\':Worker(json.loads(sys.stdin.readline())).run()\n'
worker_path=Path('/content/crew-fasth3-worker.py');worker_path.write_text(worker_source)
bridge_log_path=Path('/content/crew-fasth3-bridge.log')
if 'bridge_proc' not in globals() or bridge_proc.poll() is not None:
 bridge_log=open(bridge_log_path,'w')
 bridge_proc=subprocess.Popen([PY,'-u',str(worker_path)],stdin=subprocess.PIPE,stdout=bridge_log,stderr=subprocess.STDOUT,text=True)
 bridge_proc.stdin.write(json.dumps({'site_token':site_token,'bridge_key':bridge_key})+'\n')
 bridge_proc.stdin.flush();bridge_proc.stdin.close()
 for _ in range(45):
  output=bridge_log_path.read_text()
  if 'FASTH3_BRIDGE_ONLINE' in output:break
  if bridge_proc.poll() is not None:
   print(output[-2000:])
   raise RuntimeError('Worker 启动失败。若日志为 401/403，更新 Colab Secrets 中的两项凭据；其余错误请查看上方日志。')
  time.sleep(2)
 else:raise TimeoutError('90 秒内未确认 Worker 上线；请查看桥接日志，勿重复创建任务。')
else:print('复用已运行的 Worker，不重复启动',flush=True)
del bridge_key
print('4/4 · Worker 进程已启动。请回平台检查实时心跳；已排队任务会自动领取。',flush=True)
archive_source='"""Archive all workbench records/media to the verified Google Drive account.\nRuns on CPU; no generation privileges. Credentials are memory-only.\n"""\nimport hashlib,io,json,time,threading,mimetypes\nfrom googleapiclient.http import MediaIoBaseUpload\nTARGET=\'info@triphorium.com\'\nclass DriveArchive:\n def __init__(self,drive,cloud,site):\n  self.drive,self.cloud,self.site=drive,cloud,site\n  self.email=drive.about().get(fields=\'user(emailAddress)\').execute()[\'user\'][\'emailAddress\']\n  if self.email.lower()!=TARGET:raise RuntimeError(\'Please authenticate Google Drive as \'+TARGET)\n  self.root=self.folder(\'Triphorium Agent Records\',\'root\');self.media=self.folder(\'media\',self.root)\n  self.stop=threading.Event();self.last_hash=None;self.known={}\n def folder(self,name,parent):\n  q="trashed=false and mimeType=\'application/vnd.google-apps.folder\' and name=\'"+name+"\' and \'"+parent+"\' in parents"\n  rows=self.drive.files().list(q=q,fields=\'files(id)\',pageSize=100).execute()[\'files\']\n  return rows[0][\'id\'] if rows else self.drive.files().create(body={\'name\':name,\'mimeType\':\'application/vnd.google-apps.folder\',\'parents\':[parent]},fields=\'id\').execute()[\'id\']\n def upload(self,name,raw,parent,props=None):\n  # Stable content-derived names allow resume without duplicate uploads.\n  rows=self.drive.files().list(q="trashed=false and name=\'"+name+"\' and \'"+parent+"\' in parents",fields=\'files(id)\',pageSize=100).execute()[\'files\']\n  if rows:return rows[0][\'id\']\n  return self.drive.files().create(body={\'name\':name,\'parents\':[parent],\'appProperties\':props or {}},media_body=MediaIoBaseUpload(io.BytesIO(raw),mimetype=mimetypes.guess_type(name)[0] or \'application/octet-stream\',resumable=True),fields=\'id\').execute()[\'id\']\n def get(self,path,**kw):\n  r=self.cloud.get(self.site+\'/api/bridge/archive/\'+path,timeout=120,**kw);r.raise_for_status();return r\n def once(self):\n  state=self.get(\'state\').json();raw=json.dumps(state,ensure_ascii=False,sort_keys=True).encode();digest=hashlib.sha256(raw).hexdigest()\n  if digest!=self.last_hash:self.upload(\'records-\'+digest+\'.json\',raw,self.root);self.last_hash=digest\n  cursor=None;count=0\n  while True:\n   page=self.get(\'files\',params={\'cursor\':cursor} if cursor else {}).json()\n   for f in page[\'files\']:\n    key=f[\'key\'];version=key+\':\'+f[\'etag\'];count+=1\n    if version in self.known:continue\n    payload=self.get(key).content\n    if len(payload)!=f[\'size\']:raise RuntimeError(\'Incomplete media download\')\n    h=hashlib.sha256(payload).hexdigest();stem,ext=key.rsplit(\'.\',1);name=stem.replace(\'/\',\'_\')+\'-\'+h+\'.\'+ext\n    self.upload(name,payload,self.media,{\'sourceKey\':key,\'sha256\':h});self.known[version]=h\n   cursor=page.get(\'cursor\')\n   if not cursor:break\n  r=self.cloud.post(self.site+\'/api/bridge/archive/status\',json={\'email\':self.email,\'folderId\':self.root,\'files\':count,\'stateHash\':digest},timeout=30);r.raise_for_status()\n  print(\'DRIVE_ARCHIVE_OK\',count,\'media files\',flush=True)\n def run(self):\n  while not self.stop.is_set():\n   try:self.once()\n   except Exception as e:print(\'DRIVE_ARCHIVE_RETRY\',type(e).__name__,flush=True)\n   self.stop.wait(60)\n'
# Drive access is deferred until the user explicitly clicks the sync button.
import ipywidgets as widgets
from IPython.display import display
sync_output=widgets.Output()
sync_button=widgets.Button(description='同步到 Drive（可选）',button_style='info',layout=widgets.Layout(width='220px'))
status_button=widgets.Button(description='检查 Worker 状态',layout=widgets.Layout(width='180px'))
def sync_later(_):
 global archive_service
 sync_button.disabled=True
 with sync_output:
  sync_output.clear_output()
  try:
   from google.colab import auth
   import google.auth
   from googleapiclient.discovery import build
   auth.authenticate_user()
   google_credentials,_=google.auth.default()
   drive_api=build('drive','v3',credentials=google_credentials,cache_discovery=False)
   assert drive_api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress'].lower()=='info@triphorium.com','归档目标为 info@triphorium.com；选择该账号仅用于可选同步'
   archive_cloud=requests.Session()
   archive_cloud.headers.update({'OAI-Sites-Authorization':'Bearer '+site_token,'x-crew-archive-key':secret('CREW_ARCHIVE_KEY')})
   exec(archive_source,globals())
   if 'archive_service' in globals():archive_service.stop.set()
   archive_service=DriveArchive(drive_api,archive_cloud,SITE)
   archive_service.once()
   threading.Thread(target=archive_service.run,daemon=True).start()
   print('Drive 同步已启动；任务与素材仍保留在平台。')
  except Exception as exc:
   print('Drive 同步未启动：'+type(exc).__name__+'。请确认归档账号及 CREW_ARCHIVE_KEY；视频 Worker 不受影响。')
 sync_button.disabled=False

def check_worker(_):
 with sync_output:
  sync_output.clear_output()
  print('Worker 进程：'+('运行中' if bridge_proc.poll() is None else '已退出，请重新运行启动单元'))
  print(bridge_log_path.read_text()[-2000:] if bridge_log_path.exists() else '暂无日志')
sync_button.on_click(sync_later)
status_button.on_click(check_worker)
print('启动完成。视频保存在平台。Drive 同步未启用；只有点击下方同步按钮才请求 Drive 授权。')
print('单元执行完毕后 Worker 在此运行时后台工作。下班请断开并删除运行时，以停止 GPU 计费。')
display(widgets.HBox([status_button,sync_button]),sync_output)

